In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

csv_path   = "/home/mw/input/data_total74037403/国家角度数据集/国家.csv" 
latest     = True
target_year= 2021
top_k      = 3
use_pyvis  = False
plt.rcParams['font.family'] = 'SimHei'

dev_cols = {
    '移动网络覆盖率':'mobile_coverage',
    '固定宽带普及率':'broadband_penetration',
    '研发支出占GDP比重':'rd_gdp_ratio',
    '高等教育劳动力占比':'higher_edu_labor',
    '高等教育入学率':'higher_edu_enrollment',
    '教育支出总额占比':'edu_expenditure',
    '政府效率':'gov_efficiency',
    '研发人员（每百万人）':'rd_personnel'
}
out_cols = {          
    '独角兽公司数量':'unicorn_companies',
    '科技期刊文章':'sci_articles',
    'ICT服务出口':'ict_service_export',
    '信息通讯服务出口':'telecom_service_export',
    'ICT相关论文发表量':'ict_papers'
}


df = pd.read_csv(csv_path, encoding='utf-8')

# 1. 数值列统一转型、置 NaN
num_cols = list(dev_cols.keys()) + list(out_cols.keys())
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. 按国家-年份做线性插值，然后首尾 NaN 用前后最近值填补
df = df.sort_values(['国家名称','year'])
df[num_cols] = (
    df.groupby('国家名称')[num_cols]
      .transform(lambda g: g.interpolate(method='linear', limit_direction='both'))
)

df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

# 4. 选最新年份或指定年份
if latest:
    idx = df.groupby('国家名称')['year'].idxmax()
    df_use = df.loc[idx].copy()
else:
    df_use = df[df['year']==target_year].copy()

# 保留国家列表
countries = df_use['国家名称'].tolist()


def build_similarity(df_part, cols_dict):
    X = StandardScaler().fit_transform(df_part[list(cols_dict.keys())])
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)          # 本国与本国相似度置 0，方便取边
    return sim

def draw_network(sim_mat, cols_dict, title, node_size_ref, cmap='coolwarm'):
    G = nx.Graph()
    G.add_nodes_from(countries)

    # 为每个国家挑 top-k 相似边
    for i, c1 in enumerate(countries):
        top_idx = np.argsort(sim_mat[i])[-top_k:]
        for j in top_idx:
            c2 = countries[j]
            w  = sim_mat[i, j]
            if w > 0:                 # 防止 0 边
                G.add_edge(c1, c2, weight=w)

    # 节点大小参考列
    sizes = np.interp(df_use[node_size_ref], (df_use[node_size_ref].min(),
                                              df_use[node_size_ref].max()),
                      (400, 2000))

    # 颜色按社区划分
    try:
        import community as community_louvain
        comm = community_louvain.best_partition(G)
        colors = [comm[n] for n in G.nodes()]
    except:
        colors = 'skyblue'

    # spring 布局
    pos = nx.spring_layout(G, k=0.8, seed=42)

    # 画图
    plt.figure(figsize=(12, 9))
    edges, w = zip(*nx.get_edge_attributes(G, 'weight').items())
    nx.draw_networkx_nodes(G, pos,
                           node_size=sizes,
                           node_color=colors,
                           cmap=plt.cm.get_cmap(cmap))
    nx.draw_networkx_edges(G, pos,
                           edgelist=edges,
                           width=[4*wi for wi in w],
                           alpha=0.7)
    nx.draw_networkx_labels(G, pos,
                            font_size=10,
                            font_color='black')
    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # 交互式
    if use_pyvis:
        from pyvis.network import Network
        net = Network(height='750px', width='100%', bgcolor='#ffffff')
        for node, size in zip(G.nodes(), sizes):
            net.add_node(node, label=node, size=size)
        for u, v, d in G.edges(data=True):
            net.add_edge(u, v, value=d['weight'])
        net.force_atlas_2based()      
        net.show(title.replace(' ', '_') + '.html')


sim_dev = build_similarity(df_use, dev_cols)
draw_network(sim_dev, dev_cols,
             '科技发展相似性网络',
             node_size_ref='研发支出占GDP比重')

sim_out = build_similarity(df_use, out_cols)
draw_network(sim_out, out_cols,
             '科技成果相似性网络',
             node_size_ref='独角兽公司数量')

/opt/conda/lib/python3.6/site-packages/networkx/drawing/nx_pylab.py:579: MatplotlibDeprecationWarning: 
The iterable function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use np.iterable instead.
  if not cb.iterable(width):
findfont: Font family ['SimHei'] not found. Falling back to DejaVu Sans.
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 21457 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23637 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/o

<Figure size 1200x900 with 1 Axes>

/opt/conda/lib/python3.6/site-packages/networkx/drawing/nx_pylab.py:579: MatplotlibDeprecationWarning: 
The iterable function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use np.iterable instead.
  if not cb.iterable(width):
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25104 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 26524 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:2

<Figure size 1200x900 with 1 Axes>

In [3]:
from sklearn.cluster import KMeans

X_dev = StandardScaler().fit_transform(df_use[list(dev_cols.keys())])
k_dev = 3
kmeans_dev = KMeans(n_clusters=k_dev, random_state=42)
labels_dev = kmeans_dev.fit_predict(X_dev)
df_use['科技发展聚类'] = labels_dev

print("科技发展聚类结果：")
print(df_use[['国家名称', '科技发展聚类']])

X_out = StandardScaler().fit_transform(df_use[list(out_cols.keys())])
k_out = 3
kmeans_out = KMeans(n_clusters=k_out, random_state=42)
labels_out = kmeans_out.fit_predict(X_out)
df_use['科研成果聚类'] = labels_out

print("科研成果聚类结果：")
print(df_use[['国家名称', '科研成果聚类']])

科技发展聚类结果：
               国家名称  科技发展聚类
12        Argentina       2
25        Australia       0
38          Belgium       0
51           Brazil       2
64           Canada       0
77            China       0
90         Colombia       2
103         Denmark       0
116           Egypt       1
129         Finland       0
142          France       0
155         Germany       0
168          Greece       2
181         Iceland       0
194           India       1
207       Indonesia       1
220         Ireland       0
233          Israel       0
246           Italy       2
259           Japan       0
272           Kenya       1
285           Korea       0
298      Luxembourg       0
311        Malaysia       1
324          Mexico       1
337     Netherlands       0
350     New Zealand       0
363          Norway       0
376          Poland       2
389         Romania       2
402          Russia       2
415    Saudi Arabia       2
428       Singapore       0
441    South Africa       1
454       

In [4]:
# 科技发展聚类结果可视化
from sklearn.decomposition import PCA
plt.rcParams['axes.unicode_minus'] = False
pca = PCA(n_components=2)
X_dev_2d = pca.fit_transform(X_dev)
plt.figure(figsize=(5,5))
for i in range(k_dev):
    plt.scatter(X_dev_2d[labels_dev==i,0], X_dev_2d[labels_dev==i,1], label=f'Cluster {i}',s=200)
for i, name in enumerate(df_use['国家名称']):
    plt.text(X_dev_2d[i,0], X_dev_2d[i,1], name, fontsize=8)
plt.title('科技发展聚类')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()

findfont: Font family ['SimHei'] not found. Falling back to DejaVu Sans.
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 21457 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23637 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 32858 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.p

<Figure size 360x360 with 1 Axes>

In [5]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from networkx.algorithms.community.quality import modularity

# 科技发展网络
sim_dev = build_similarity(df_use, dev_cols)
G_dev = nx.Graph()
G_dev.add_nodes_from(countries)
for i, c1 in enumerate(countries):
    top_idx = np.argsort(sim_dev[i])[-top_k:]
    for j in top_idx:
        c2 = countries[j]
        w  = sim_dev[i, j]
        if w > 0:
            G_dev.add_edge(c1, c2, weight=w)

# KMeans标签
kmeans_labels_dev = df_use['科技发展聚类'].values

# 将KMeans标签转为社群集合
from collections import defaultdict
partition_dev = defaultdict(list)
for node, cid in zip(countries, kmeans_labels_dev):
    partition_dev[cid].append(node)
partition_dev = [set(nodes) for nodes in partition_dev.values()]

# 计算模块度
mod_dev = modularity(G_dev, partition_dev, weight='weight')
# 由于KMeans本身就是标签，NMI和ARI与自身比为1
nmi_dev = normalized_mutual_info_score(kmeans_labels_dev, kmeans_labels_dev)
ari_dev = adjusted_rand_score(kmeans_labels_dev, kmeans_labels_dev)

print(f"KMeans-科技发展网络 NMI: {nmi_dev:.4f}, ARI: {ari_dev:.4f}, 最佳模块度: {mod_dev:.4f}")

# 科研成果网络
sim_out = build_similarity(df_use, out_cols)
G_out = nx.Graph()
G_out.add_nodes_from(countries)
for i, c1 in enumerate(countries):
    top_idx = np.argsort(sim_out[i])[-top_k:]
    for j in top_idx:
        c2 = countries[j]
        w  = sim_out[i, j]
        if w > 0:
            G_out.add_edge(c1, c2, weight=w)

kmeans_labels_out = df_use['科研成果聚类'].values
partition_out = defaultdict(list)
for node, cid in zip(countries, kmeans_labels_out):
    partition_out[cid].append(node)
partition_out = [set(nodes) for nodes in partition_out.values()]
mod_out = modularity(G_out, partition_out, weight='weight')
nmi_out = normalized_mutual_info_score(kmeans_labels_out, kmeans_labels_out)
ari_out = adjusted_rand_score(kmeans_labels_out, kmeans_labels_out)

print(f"KMeans-科研成果网络 NMI: {nmi_out:.4f}, ARI: {ari_out:.4f}, 最佳模块度: {mod_out:.4f}")

KMeans-科技发展网络 NMI: 1.0000, ARI: 1.0000, 最佳模块度: 0.4270
KMeans-科研成果网络 NMI: 1.0000, ARI: 1.0000, 最佳模块度: 0.1582


/opt/conda/lib/python3.6/site-packages/sklearn/metrics/cluster/supervised.py:859: FutureWarning: The behavior of NMI will change in version 0.22. To match the behavior of 'v_measure_score', NMI will use average_method='arithmetic' by default.
  FutureWarning)
/opt/conda/lib/python3.6/site-packages/sklearn/metrics/cluster/supervised.py:859: FutureWarning: The behavior of NMI will change in version 0.22. To match the behavior of 'v_measure_score', NMI will use average_method='arithmetic' by default.
  FutureWarning)


In [6]:
from sklearn.decomposition import PCA

pca_out = PCA(n_components=2)
X_out_2d = pca_out.fit_transform(X_out)
plt.figure(figsize=(8,6))
for i in range(k_out):
    plt.scatter(X_out_2d[labels_out==i,0], X_out_2d[labels_out==i,1], label=f'Cluster {i}')
for i, name in enumerate(df_use['国家名称']):
    plt.text(X_out_2d[i,0], X_out_2d[i,1], name, fontsize=8)
plt.title('科研成果聚类（PCA降维）')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend()
plt.show()

/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 30740 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25104 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 26524 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 32858 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31867 missing from current font.
  font.set_

<Figure size 576x432 with 1 Axes>

In [7]:
def draw_network(sim_mat, cols_dict, title, node_size_ref, cmap='coolwarm'):
    G = nx.Graph()
    G.add_nodes_from(countries)

    # 为每个国家挑 top-k 相似边
    for i, c1 in enumerate(countries):
        top_idx = np.argsort(sim_mat[i])[-top_k:]
        for j in top_idx:
            c2 = countries[j]
            w  = sim_mat[i, j]
            if w > 0:
                G.add_edge(c1, c2, weight=w)

    # 节点大小参考列
    sizes = np.interp(df_use[node_size_ref], (df_use[node_size_ref].min(),
                                              df_use[node_size_ref].max()),
                      (400, 2000))

    # Louvain社群识别
    try:
        import community as community_louvain
        comm = community_louvain.best_partition(G)
        colors = [comm[n] for n in G.nodes()]
        # 输出社群划分结果
        print(f"{title} Louvain社群划分：")
        for cid in set(comm.values()):
            members = [node for node, c in comm.items() if c == cid]
            print(f"社群{cid}: {members}")
    except Exception as e:
        colors = 'skyblue'
        print("Louvain社群识别失败：", e)

    # spring 布局
    pos = nx.spring_layout(G, k=0.8, seed=42)

    # 画图
    plt.figure(figsize=(12, 9))
    edges, w = zip(*nx.get_edge_attributes(G, 'weight').items())
    nx.draw_networkx_nodes(G, pos,
                           node_size=sizes,
                           node_color=colors,
                           cmap=plt.cm.get_cmap(cmap))
    nx.draw_networkx_edges(G, pos,
                           edgelist=edges,
                           width=[4*wi for wi in w],
                           alpha=0.7)
    nx.draw_networkx_labels(G, pos,
                            font_size=10,
                            font_color='black')
    plt.title(title, fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # 交互式
    if use_pyvis:
        from pyvis.network import Network
        net = Network(height='750px', width='100%', bgcolor='#ffffff')
        for node, size in zip(G.nodes(), sizes):
            net.add_node(node, label=node, size=size)
        for u, v, d in G.edges(data=True):
            net.add_edge(u, v, value=d['weight'])
        net.force_atlas_2based()
        net.show(title.replace(' ', '_') + '.html')


In [8]:
sim_dev = build_similarity(df_use, dev_cols)
draw_network(sim_dev, dev_cols, '科技发展相似性网络', node_size_ref='研发支出占GDP比重')

sim_out = build_similarity(df_use, out_cols)
draw_network(sim_out, out_cols, '科技成果相似性网络', node_size_ref='独角兽公司数量')

科技发展相似性网络 Louvain社群划分：
社群0: ['Australia', 'Ireland', 'New Zealand', 'Norway', 'Saudi Arabia', 'Singapore', 'United Kingdom', 'Uruguay']
社群1: ['Egypt', 'India', 'Kenya', 'Russia']
社群2: ['Belgium', 'Finland', 'Israel', 'Korea', 'United States']
社群3: ['Brazil', 'Colombia', 'Indonesia', 'Malaysia', 'Mexico', 'South Africa']
社群4: ['Argentina', 'Greece', 'Italy', 'Poland', 'Romania', 'Spain', 'Thailand', 'Turkiye']
社群5: ['Canada', 'China', 'Denmark', 'France', 'Germany', 'Iceland', 'Japan', 'Luxembourg', 'Netherlands', 'Sweden', 'Switzerland']


/opt/conda/lib/python3.6/site-packages/networkx/drawing/nx_pylab.py:579: MatplotlibDeprecationWarning: 
The iterable function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use np.iterable instead.
  if not cb.iterable(width):
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 21457 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23637 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:2

<Figure size 864x648 with 1 Axes>

科技成果相似性网络 Louvain社群划分：
社群0: ['Argentina', 'Kenya', 'Romania', 'Uruguay']
社群1: ['Australia', 'Colombia', 'Egypt', 'Greece']
社群2: ['Belgium', 'Canada', 'France', 'Japan', 'Korea', 'Netherlands', 'Poland', 'Switzerland']
社群3: ['Brazil', 'Iceland', 'Indonesia', 'Malaysia', 'New Zealand', 'Norway', 'Russia', 'Saudi Arabia', 'Singapore']
社群4: ['China', 'Germany', 'United Kingdom', 'United States']
社群5: ['Denmark', 'Italy', 'Luxembourg', 'Mexico', 'South Africa', 'Spain', 'Thailand', 'Turkiye']
社群6: ['Finland', 'India', 'Ireland', 'Israel', 'Sweden']


/opt/conda/lib/python3.6/site-packages/networkx/drawing/nx_pylab.py:579: MatplotlibDeprecationWarning: 
The iterable function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use np.iterable instead.
  if not cb.iterable(width):
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25104 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 26524 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:2

<Figure size 864x648 with 1 Axes>

In [9]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from networkx.algorithms.community.quality import modularity

# Louvain社群识别
import community as community_louvain

sim_dev = build_similarity(df_use, dev_cols)
G = nx.Graph()
G.add_nodes_from(countries)
for i, c1 in enumerate(countries):
    top_idx = np.argsort(sim_dev[i])[-top_k:]
    for j in top_idx:
        c2 = countries[j]
        w  = sim_dev[i, j]
        if w > 0:
            G.add_edge(c1, c2, weight=w)

comm = community_louvain.best_partition(G, weight='weight')
louvain_labels = np.array([comm[n] for n in G.nodes()])
true_labels = df_use['科技发展聚类'].values

# NMI和ARI
nmi = normalized_mutual_info_score(true_labels, louvain_labels)
ari = adjusted_rand_score(true_labels, louvain_labels)
print(f"Louvain NMI: {nmi:.4f}")
print(f"Louvain ARI: {ari:.4f}")

# Louvain模块度
# 需将partition转为list of sets
from collections import defaultdict
partition = defaultdict(list)
for node, cid in comm.items():
    partition[cid].append(node)
partition = [set(nodes) for nodes in partition.values()]
mod = modularity(G, partition, weight='weight')
print(f"Louvain 最佳模块度: {mod:.4f}")

Louvain NMI: 0.5028
Louvain ARI: 0.2775
Louvain 最佳模块度: 0.6095


/opt/conda/lib/python3.6/site-packages/sklearn/metrics/cluster/supervised.py:859: FutureWarning: The behavior of NMI will change in version 0.22. To match the behavior of 'v_measure_score', NMI will use average_method='arithmetic' by default.
  FutureWarning)


In [10]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from networkx.algorithms.community.quality import modularity
import community as community_louvain

# 科研成果网络 Louvain社群识别
sim_out = build_similarity(df_use, out_cols)
G_out = nx.Graph()
G_out.add_nodes_from(countries)
for i, c1 in enumerate(countries):
    top_idx = np.argsort(sim_out[i])[-top_k:]
    for j in top_idx:
        c2 = countries[j]
        w  = sim_out[i, j]
        if w > 0:
            G_out.add_edge(c1, c2, weight=w)

comm_out = community_louvain.best_partition(G_out, weight='weight')
louvain_labels_out = np.array([comm_out[n] for n in G_out.nodes()])
true_labels_out = df_use['科研成果聚类'].values

# NMI和ARI
nmi_out = normalized_mutual_info_score(true_labels_out, louvain_labels_out)
ari_out = adjusted_rand_score(true_labels_out, louvain_labels_out)
print(f"Louvain NMI（科研成果网络）: {nmi_out:.4f}")
print(f"Louvain ARI（科研成果网络）: {ari_out:.4f}")

# Louvain模块度
from collections import defaultdict
partition_out = defaultdict(list)
for node, cid in comm_out.items():
    partition_out[cid].append(node)
partition_out = [set(nodes) for nodes in partition_out.values()]
mod_out = modularity(G_out, partition_out, weight='weight')
print(f"Louvain 最佳模块度（科研成果网络）: {mod_out:.4f}")

Louvain NMI（科研成果网络）: 0.4752
Louvain ARI（科研成果网络）: 0.1132
Louvain 最佳模块度（科研成果网络）: 0.6834


/opt/conda/lib/python3.6/site-packages/sklearn/metrics/cluster/supervised.py:859: FutureWarning: The behavior of NMI will change in version 0.22. To match the behavior of 'v_measure_score', NMI will use average_method='arithmetic' by default.
  FutureWarning)


girvan_newman 社群识别

In [11]:
from networkx.algorithms.community import girvan_newman

def draw_network_with_gn(sim_mat, cols_dict, title, node_size_ref, cmap='coolwarm', n_communities=3):
    G = nx.Graph()
    G.add_nodes_from(countries)

    # 为每个国家挑 top-k 相似边
    for i, c1 in enumerate(countries):
        top_idx = np.argsort(sim_mat[i])[-top_k:]
        for j in top_idx:
            c2 = countries[j]
            w  = sim_mat[i, j]
            if w > 0:
                G.add_edge(c1, c2, weight=w)

    # 节点大小参考列
    sizes = np.interp(df_use[node_size_ref], (df_use[node_size_ref].min(),
                                              df_use[node_size_ref].max()),
                      (400, 2000))

    # Girvan-Newman社群识别
    comp = girvan_newman(G)
    limited = tuple(sorted(c) for c in next(comp))
    for _ in range(n_communities-2):
        limited = tuple(sorted(c) for c in next(comp))
    # 输出社群划分结果
    print(f"{title} Girvan-Newman社群划分：")
    for i, comm in enumerate(limited):
        print(f"社群{i}: {list(comm)}")
    # 颜色映射
    node_color_map = {}
    for i, comm in enumerate(limited):
        for node in comm:
            node_color_map[node] = i
    colors = [node_color_map[n] for n in G.nodes()]

    # spring 布局
    pos = nx.spring_layout(G, k=0.8, seed=42)

    # 画图
    plt.figure(figsize=(12, 9))
    edges, w = zip(*nx.get_edge_attributes(G, 'weight').items())
    nx.draw_networkx_nodes(G, pos,
                           node_size=sizes,
                           node_color=colors,
                           cmap=plt.cm.get_cmap(cmap))
    nx.draw_networkx_edges(G, pos,
                           edgelist=edges,
                           width=[4*wi for wi in w],
                           alpha=0.7)
    nx.draw_networkx_labels(G, pos,
                            font_size=10,
                            font_color='black')
    plt.title(title + "（Girvan-Newman社群）", fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

sim_dev = build_similarity(df_use, dev_cols)
draw_network_with_gn(sim_dev, dev_cols, '科技发展相似性网络', node_size_ref='研发支出占GDP比重', n_communities=3)

sim_out = build_similarity(df_use, out_cols)
draw_network_with_gn(sim_out, out_cols, '科技成果相似性网络', node_size_ref='独角兽公司数量', n_communities=3)

科技发展相似性网络 Girvan-Newman社群划分：
社群0: ['Argentina', 'Brazil', 'Colombia', 'Egypt', 'India', 'Indonesia', 'Italy', 'Kenya', 'Malaysia', 'Mexico', 'Poland', 'Romania', 'Russia', 'Saudi Arabia', 'South Africa', 'Thailand', 'Turkiye', 'Uruguay']
社群1: ['Australia', 'Greece', 'Ireland', 'New Zealand', 'Norway', 'Singapore', 'Spain', 'United Kingdom']
社群2: ['Belgium', 'Canada', 'China', 'Denmark', 'Finland', 'France', 'Germany', 'Iceland', 'Israel', 'Japan', 'Korea', 'Luxembourg', 'Netherlands', 'Sweden', 'Switzerland', 'United States']


/opt/conda/lib/python3.6/site-packages/networkx/drawing/nx_pylab.py:579: MatplotlibDeprecationWarning: 
The iterable function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use np.iterable instead.
  if not cb.iterable(width):
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 21457 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23637 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:2

<Figure size 864x648 with 1 Axes>

科技成果相似性网络 Girvan-Newman社群划分：
社群0: ['Argentina', 'Finland', 'India', 'Ireland', 'Israel', 'Kenya', 'Romania', 'Sweden', 'Uruguay']
社群1: ['Australia', 'Brazil', 'Colombia', 'Denmark', 'Egypt', 'Greece', 'Iceland', 'Indonesia', 'Italy', 'Luxembourg', 'Malaysia', 'Mexico', 'New Zealand', 'Norway', 'Russia', 'Saudi Arabia', 'Singapore', 'South Africa', 'Spain', 'Thailand', 'Turkiye']
社群2: ['Belgium', 'Canada', 'China', 'France', 'Germany', 'Japan', 'Korea', 'Netherlands', 'Poland', 'Switzerland', 'United Kingdom', 'United States']


/opt/conda/lib/python3.6/site-packages/networkx/drawing/nx_pylab.py:579: MatplotlibDeprecationWarning: 
The iterable function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use np.iterable instead.
  if not cb.iterable(width):
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25104 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 26524 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:2

<Figure size 864x648 with 1 Axes>

In [12]:
from networkx.algorithms.community import girvan_newman
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from networkx.algorithms.community.quality import modularity
from collections import defaultdict

def gn_metrics(sim_mat, true_labels, n_communities=3):
    G = nx.Graph()
    G.add_nodes_from(countries)
    for i, c1 in enumerate(countries):
        top_idx = np.argsort(sim_mat[i])[-top_k:]
        for j in top_idx:
            c2 = countries[j]
            w  = sim_mat[i, j]
            if w > 0:
                G.add_edge(c1, c2, weight=w)
    # GN社群划分
    comp = girvan_newman(G)
    limited = tuple(sorted(c) for c in next(comp))
    for _ in range(n_communities-2):
        limited = tuple(sorted(c) for c in next(comp))
    # 转为标签
    node2idx = {name: idx for idx, name in enumerate(countries)}
    gn_labels = np.zeros(len(countries), dtype=int)
    for cid, comm in enumerate(limited):
        for node in comm:
            gn_labels[node2idx[node]] = cid
    # NMI, ARI
    nmi = normalized_mutual_info_score(true_labels, gn_labels)
    ari = adjusted_rand_score(true_labels, gn_labels)
    # GN模块度
    comm_sets = [set(comm) for comm in limited]
    mod = modularity(G, comm_sets, weight='weight')
    return nmi, ari, mod

# 科技发展网络
sim_dev = build_similarity(df_use, dev_cols)
true_labels_dev = df_use['科技发展聚类'].values
nmi_dev, ari_dev, mod_dev = gn_metrics(sim_dev, true_labels_dev, n_communities=3)
print(f"GN-科技发展网络 NMI: {nmi_dev:.4f}, ARI: {ari_dev:.4f}, 最佳模块度: {mod_dev:.4f}")

# 科技成果网络
sim_out = build_similarity(df_use, out_cols)
true_labels_out = df_use['科研成果聚类'].values
nmi_out, ari_out, mod_out = gn_metrics(sim_out, true_labels_out, n_communities=3)
print(f"GN-科技成果网络 NMI: {nmi_out:.4f}, ARI: {ari_out:.4f}, 最佳模块度: {mod_out:.4f}")

GN-科技发展网络 NMI: 0.6183, ARI: 0.4867, 最佳模块度: 0.5278
GN-科技成果网络 NMI: 0.3636, ARI: 0.2189, 最佳模块度: 0.5546


/opt/conda/lib/python3.6/site-packages/sklearn/metrics/cluster/supervised.py:859: FutureWarning: The behavior of NMI will change in version 0.22. To match the behavior of 'v_measure_score', NMI will use average_method='arithmetic' by default.
  FutureWarning)
/opt/conda/lib/python3.6/site-packages/sklearn/metrics/cluster/supervised.py:859: FutureWarning: The behavior of NMI will change in version 0.22. To match the behavior of 'v_measure_score', NMI will use average_method='arithmetic' by default.
  FutureWarning)


Label Propagation社群识别

In [13]:
from networkx.algorithms.community import asyn_lpa_communities

def draw_network_with_label_propagation(sim_mat, cols_dict, title, node_size_ref, cmap='coolwarm'):
    G = nx.Graph()
    G.add_nodes_from(countries)

    # 为每个国家挑 top-k 相似边
    for i, c1 in enumerate(countries):
        top_idx = np.argsort(sim_mat[i])[-top_k:]
        for j in top_idx:
            c2 = countries[j]
            w  = sim_mat[i, j]
            if w > 0:
                G.add_edge(c1, c2, weight=w)

    # 节点大小参考列
    sizes = np.interp(df_use[node_size_ref], (df_use[node_size_ref].min(),
                                              df_use[node_size_ref].max()),
                      (400, 2000))

    # Label Propagation社群识别
    comms = list(asyn_lpa_communities(G, weight='weight', seed=42))
    print(f"{title} Label Propagation社群划分：")
    for i, comm in enumerate(comms):
        print(f"社群{i}: {list(comm)}")
    # 颜色映射
    node_color_map = {}
    for i, comm in enumerate(comms):
        for node in comm:
            node_color_map[node] = i
    colors = [node_color_map[n] for n in G.nodes()]

    # spring 布局
    pos = nx.spring_layout(G, k=0.8, seed=42)

    # 画图
    plt.figure(figsize=(12, 9))
    edges, w = zip(*nx.get_edge_attributes(G, 'weight').items())
    nx.draw_networkx_nodes(G, pos,
                           node_size=sizes,
                           node_color=colors,
                           cmap=plt.cm.get_cmap(cmap))
    nx.draw_networkx_edges(G, pos,
                           edgelist=edges,
                           width=[4*wi for wi in w],
                           alpha=0.7)
    nx.draw_networkx_labels(G, pos,
                            font_size=10,
                            font_color='black')
    plt.title(title + "（Label Propagation社群）", fontsize=16)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

sim_dev = build_similarity(df_use, dev_cols)
draw_network_with_label_propagation(sim_dev, dev_cols, '科技发展相似性网络', node_size_ref='研发支出占GDP比重')

sim_out = build_similarity(df_use, out_cols)
draw_network_with_label_propagation(sim_out, out_cols, '科技成果相似性网络', node_size_ref='独角兽公司数量')

科技发展相似性网络 Label Propagation社群划分：
社群0: ['Argentina', 'Turkiye', 'Greece']
社群1: ['United Kingdom', 'Norway', 'Japan', 'Australia', 'New Zealand', 'Ireland', 'Singapore']
社群2: ['United States', 'Israel', 'Korea', 'Belgium', 'Finland']
社群3: ['Colombia', 'Indonesia', 'Brazil', 'Mexico']
社群4: ['Iceland', 'Denmark', 'Luxembourg', 'Netherlands', 'Sweden', 'Germany', 'Switzerland', 'Canada']
社群5: ['China', 'France']
社群6: ['Egypt', 'Kenya', 'India']
社群7: ['Italy']
社群8: ['South Africa', 'Malaysia']
社群9: ['Romania', 'Poland', 'Thailand']
社群10: ['Russia']
社群11: ['Uruguay', 'Saudi Arabia']
社群12: ['Spain']


/opt/conda/lib/python3.6/site-packages/networkx/drawing/nx_pylab.py:579: MatplotlibDeprecationWarning: 
The iterable function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use np.iterable instead.
  if not cb.iterable(width):
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 21457 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 23637 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:2

<Figure size 864x648 with 1 Axes>

科技成果相似性网络 Label Propagation社群划分：
社群0: ['Argentina', 'Uruguay', 'Kenya']
社群1: ['Colombia', 'Egypt', 'Greece', 'Australia']
社群2: ['United Kingdom', 'Japan', 'Netherlands', 'Korea', 'Poland', 'Belgium', 'Switzerland', 'France', 'Canada']
社群3: ['Brazil']
社群4: ['China', 'United States', 'Germany']
社群5: ['Denmark', 'Luxembourg', 'South Africa', 'Thailand', 'Mexico', 'Turkiye']
社群6: ['Israel', 'India', 'Ireland', 'Sweden', 'Finland']
社群7: ['Iceland', 'Norway', 'Malaysia', 'Indonesia', 'New Zealand']
社群8: ['Russia', 'Spain', 'Italy']
社群9: ['Romania']
社群10: ['Singapore', 'Saudi Arabia']


/opt/conda/lib/python3.6/site-packages/networkx/drawing/nx_pylab.py:579: MatplotlibDeprecationWarning: 
The iterable function was deprecated in Matplotlib 3.1 and will be removed in 3.3. Use np.iterable instead.
  if not cb.iterable(width):
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 31185 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25216 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 25104 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:211: RuntimeWarning: Glyph 26524 missing from current font.
  font.set_text(s, 0.0, flags=flags)
/opt/conda/lib/python3.6/site-packages/matplotlib/backends/backend_agg.py:2

<Figure size 864x648 with 1 Axes>

In [14]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from networkx.algorithms.community.quality import modularity

# 以科技发展网络为例
sim_dev = build_similarity(df_use, dev_cols)
G = nx.Graph()
G.add_nodes_from(countries)
for i, c1 in enumerate(countries):
    top_idx = np.argsort(sim_dev[i])[-top_k:]
    for j in top_idx:
        c2 = countries[j]
        w  = sim_dev[i, j]
        if w > 0:
            G.add_edge(c1, c2, weight=w)

# Label Propagation社群
from networkx.algorithms.community import asyn_lpa_communities
comms = list(asyn_lpa_communities(G, weight='weight', seed=42))
# 生成节点到社群的标签
lp_labels = np.zeros(len(countries), dtype=int)
node2idx = {name: idx for idx, name in enumerate(countries)}
for cid, comm in enumerate(comms):
    for node in comm:
        lp_labels[node2idx[node]] = cid

# 真实标签（如KMeans聚类结果）
true_labels = df_use['科技发展聚类'].values

# NMI和ARI
nmi = normalized_mutual_info_score(true_labels, lp_labels)
ari = adjusted_rand_score(true_labels, lp_labels)
print(f"Label Propagation NMI: {nmi:.4f}")
print(f"Label Propagation ARI: {ari:.4f}")

# 最佳模块度
mod = modularity(G, comms, weight='weight')
print(f"Label Propagation 最佳模块度: {mod:.4f}")

Label Propagation NMI: 0.6033
Label Propagation ARI: 0.2392
Label Propagation 最佳模块度: 0.4883


/opt/conda/lib/python3.6/site-packages/sklearn/metrics/cluster/supervised.py:859: FutureWarning: The behavior of NMI will change in version 0.22. To match the behavior of 'v_measure_score', NMI will use average_method='arithmetic' by default.
  FutureWarning)


In [15]:
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from networkx.algorithms.community.quality import modularity

# 科研成果网络
sim_out = build_similarity(df_use, out_cols)
G_out = nx.Graph()
G_out.add_nodes_from(countries)
for i, c1 in enumerate(countries):
    top_idx = np.argsort(sim_out[i])[-top_k:]
    for j in top_idx:
        c2 = countries[j]
        w  = sim_out[i, j]
        if w > 0:
            G_out.add_edge(c1, c2, weight=w)

# Label Propagation社群
comms_out = list(asyn_lpa_communities(G_out, weight='weight', seed=42))
lp_labels_out = np.zeros(len(countries), dtype=int)
node2idx_out = {name: idx for idx, name in enumerate(countries)}
for cid, comm in enumerate(comms_out):
    for node in comm:
        lp_labels_out[node2idx_out[node]] = cid

# 真实标签（如KMeans聚类结果）
true_labels_out = df_use['科研成果聚类'].values

# NMI和ARI
nmi_out = normalized_mutual_info_score(true_labels_out, lp_labels_out)
ari_out = adjusted_rand_score(true_labels_out, lp_labels_out)
print(f"Label Propagation NMI（科研成果网络）: {nmi_out:.4f}")
print(f"Label Propagation ARI（科研成果网络）: {ari_out:.4f}")

# 最佳模块度
mod_out = modularity(G_out, comms_out, weight='weight')
print(f"Label Propagation 最佳模块度（科研成果网络）: {mod_out:.4f}")

Label Propagation NMI（科研成果网络）: 0.4560
Label Propagation ARI（科研成果网络）: 0.0833
Label Propagation 最佳模块度（科研成果网络）: 0.5814


/opt/conda/lib/python3.6/site-packages/sklearn/metrics/cluster/supervised.py:859: FutureWarning: The behavior of NMI will change in version 0.22. To match the behavior of 'v_measure_score', NMI will use average_method='arithmetic' by default.
  FutureWarning)
